## 1) Libraries

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor, XGBClassifier

import warnings
warnings.filterwarnings("ignore")

## 2) Data

In [ ]:
data_raw = pd.read_csv("train.csv")
test_data_raw = pd.read_csv("test.csv")

In [ ]:
def clean_data(df):
    df = df.copy()

    # --------------------------------------------------
    # Clean price column if present
    # Only training data has price
    # --------------------------------------------------
    if "price" in df.columns:
        df["price"] = (
            df["price"]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df["price"] = pd.to_numeric(df["price"], errors="coerce")

        df["is_placeholder_price"] = (
            (df["price"] >= 30000) |
            (df["price"].between(39000, 41000)) |
            (df["price"].between(49000, 51000))
        ).astype(int)

    # --------------------------------------------------
    # Clean percentage columns
    # --------------------------------------------------
    percent_cols = ["host_response_rate", "host_acceptance_rate"]

    for col in percent_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .replace("nan", np.nan)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --------------------------------------------------
    # Convert yes/no columns to 1/0
    # --------------------------------------------------
    yn_cols = [
        "host_is_superhost",
        "host_has_profile_pic",
        "host_identity_verified",
        "instant_bookable",
        "has_availability"
    ]

    for col in yn_cols:
        if col in df.columns:
            df[col] = df[col].map({
                "t": 1, "f": 0,
                "yes": 1, "no": 0,
                True: 1, False: 0
            })

    # --------------------------------------------------
    # Date features
    # --------------------------------------------------
    date_cols = ["host_since", "first_review", "last_review"]

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    reference_year = 2026

    if "host_since" in df.columns:
        df["host_years"] = reference_year - df["host_since"].dt.year
        df["host_since_missing"] = df["host_since"].isna().astype(int)

    if "first_review" in df.columns:
        df["years_since_first_review"] = reference_year - df["first_review"].dt.year
        df["first_review_missing"] = df["first_review"].isna().astype(int)

    if "last_review" in df.columns:
        df["years_since_last_review"] = reference_year - df["last_review"].dt.year
        df["last_review_missing"] = df["last_review"].isna().astype(int)

    df = df.drop(columns=[col for col in date_cols if col in df.columns])

    # --------------------------------------------------
    # Hotel routing feature
    # This is the key Junho-inspired variable
    # --------------------------------------------------
    if "property_type" in df.columns:
        prop = df["property_type"].fillna("").astype(str).str.lower()

        hotel_words = [
            "hotel",
            "boutique hotel",
            "resort",
            "hostel",
            "serviced apartment",
            "aparthotel"
        ]

        df["is_hotel"] = prop.apply(
            lambda x: int(any(word in x for word in hotel_words))
        )
    else:
        df["is_hotel"] = 0

    # Optional backup using name/description if available
    if "name" in df.columns:
        name_lower = df["name"].fillna("").astype(str).str.lower()
        df["name_mentions_hotel"] = name_lower.str.contains(
            "hotel|resort|inn|suite|boutique", 
            regex=True
        ).astype(int)

        df["is_hotel"] = ((df["is_hotel"] == 1) | (df["name_mentions_hotel"] == 1)).astype(int)

    # --------------------------------------------------
    # Amenities features
    # --------------------------------------------------
    if "amenities" in df.columns:
        amenities_text = df["amenities"].fillna("").astype(str)

        # Fix: empty amenities should be 0, not 1
        df["amenities_count"] = np.where(
            amenities_text.str.strip().eq(""),
            0,
            amenities_text.str.count(",") + 1
        )

        important_amenities = [
            "Wifi",
            "Air conditioning",
            "Kitchen",
            "Washer",
            "Dryer",
            "Free parking",
            "Pool",
            "Hot tub",
            "Gym",
            "Elevator",
            "Pets allowed",
            "Self check-in",
            "Dedicated workspace",
            "TV",
            "Heating",
            "Patio",
            "Backyard",
            "BBQ grill"
        ]

        for amenity in important_amenities:
            new_col = "has_" + amenity.lower().replace(" ", "_").replace("-", "_")

            df[new_col] = (
                amenities_text
                .str.contains(amenity, case=False, regex=False)
                .astype(int)
            )

    # --------------------------------------------------
    # Text length features
    # --------------------------------------------------
    text_cols = ["description", "host_about", "name", "neighborhood_overview"]

    for col in text_cols:
        if col in df.columns:
            df[col + "_length"] = df[col].fillna("").astype(str).str.len()
            df[col + "_word_count"] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.split()
                .str.len()
            )

    # --------------------------------------------------
    # Bathroom count
    # --------------------------------------------------
    if "bathrooms_text" in df.columns:
        bath_text = df["bathrooms_text"].fillna("").astype(str).str.lower()

        df["bathrooms_num"] = (
            bath_text
            .str.extract(r"(\d+\.?\d*)")[0]
        )

        df["bathrooms_num"] = pd.to_numeric(df["bathrooms_num"], errors="coerce")

        df.loc[
            bath_text.str.contains("half", na=False),
            "bathrooms_num"
        ] = 0.5

        df["shared_bath"] = bath_text.str.contains("shared", na=False).astype(int)
        df["private_bath"] = bath_text.str.contains("private", na=False).astype(int)

    # --------------------------------------------------
    # Ratio features
    # --------------------------------------------------
    if "accommodates" in df.columns and "bedrooms" in df.columns:
        df["accommodates_per_bedroom"] = df["accommodates"] / (df["bedrooms"] + 1)

    if "beds" in df.columns and "bedrooms" in df.columns:
        df["beds_per_bedroom"] = df["beds"] / (df["bedrooms"] + 1)

    if "bathrooms_num" in df.columns and "accommodates" in df.columns:
        df["bathrooms_per_guest"] = df["bathrooms_num"] / (df["accommodates"] + 1)

    if "beds" in df.columns and "accommodates" in df.columns:
        df["beds_per_guest"] = df["beds"] / (df["accommodates"] + 1)

    if "number_of_reviews" in df.columns and "host_years" in df.columns:
        df["reviews_per_host_year"] = df["number_of_reviews"] / (df["host_years"] + 1)

    if "number_of_reviews" in df.columns and "years_since_first_review" in df.columns:
        df["reviews_per_year_since_first_review"] = (
            df["number_of_reviews"] / (df["years_since_first_review"] + 1)
        )

    # --------------------------------------------------
    # Missingness indicators for important numeric columns
    # Tree models can use these
    # --------------------------------------------------
    missing_indicator_cols = [
        "bedrooms",
        "beds",
        "bathrooms_num",
        "review_scores_rating",
        "host_response_rate",
        "host_acceptance_rate",
        "host_listings_count",
        "host_total_listings_count"
    ]

    for col in missing_indicator_cols:
        if col in df.columns:
            df[col + "_missing"] = df[col].isna().astype(int)

    # --------------------------------------------------
    # Replace infinite values caused by divisions
    # --------------------------------------------------
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [ ]:
data_clean = clean_data(data_raw)
test_data_clean = clean_data(test_data_raw)

train_prices = data_clean["price"]
test_ids = test_data_raw["id"]

drop_cols = [
    "price",
    "is_placeholder_price"
]

X_data_clean = data_clean.drop(
    columns=[col for col in drop_cols if col in data_clean.columns]
)

X_test_data_clean = test_data_clean.drop(
    columns=[col for col in drop_cols if col in test_data_clean.columns]
)

if "id" in X_data_clean.columns:
    X_data_clean = X_data_clean.drop(columns=["id"])

if "id" in X_test_data_clean.columns:
    X_test_data_clean = X_test_data_clean.drop(columns=["id"])

raw_text_cols = [
    "description",
    "host_about",
    "amenities",
    "name",
    "neighborhood_overview",
    "host_verifications"
]

X_data_clean = X_data_clean.drop(
    columns=[col for col in raw_text_cols if col in X_data_clean.columns]
)

X_test_data_clean = X_test_data_clean.drop(
    columns=[col for col in raw_text_cols if col in X_test_data_clean.columns]
)

X_test_data_clean = X_test_data_clean.reindex(columns=X_data_clean.columns)

combined_X = pd.concat(
    [X_data_clean, X_test_data_clean],
    axis=0
)

cat_cols = combined_X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    combined_X[col] = combined_X[col].fillna("Missing").astype(str)

combined_X_encoded = pd.get_dummies(
    combined_X,
    columns=cat_cols,
    dummy_na=False
)

combined_X_encoded.columns = (
    combined_X_encoded.columns
    .astype(str)
    .str.replace("[", "(", regex=False)
    .str.replace("]", ")", regex=False)
    .str.replace("<", "_", regex=False)
)

X_encoded = combined_X_encoded.iloc[:len(X_data_clean), :].copy()
X_test_encoded = combined_X_encoded.iloc[len(X_data_clean):, :].copy()

X_encoded = X_encoded.replace([np.inf, -np.inf], np.nan)
X_test_encoded = X_test_encoded.replace([np.inf, -np.inf], np.nan)

## 3) Machine Learning Model

In [ ]:
full_model_data = X_encoded.copy()
full_model_data["price"] = train_prices

nonhotel_full_data = full_model_data[
    full_model_data["is_hotel"] == 0
].copy()

hotel_full_data = full_model_data[
    full_model_data["is_hotel"] == 1
].copy()

real_hotel_full_data = hotel_full_data[
    hotel_full_data["price"] < 30000
].copy()

placeholder_hotel_full_data = hotel_full_data[
    hotel_full_data["price"] >= 30000
].copy()

In [ ]:
X_nonhotel_full = nonhotel_full_data.drop(columns=["price"])
y_nonhotel_full = nonhotel_full_data["price"]

final_nonhotel_model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.02,
    max_depth=5,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    reg_alpha=0.1,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=12,
    tree_method="hist",
    n_jobs=-1
)

final_nonhotel_model.fit(
    X_nonhotel_full,
    y_nonhotel_full,
    verbose=False
)

In [ ]:
X_real_hotel_full = real_hotel_full_data.drop(columns=["price"])
y_real_hotel_full = real_hotel_full_data["price"]

final_real_hotel_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    reg_alpha=0.1,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=12,
    tree_method="hist",
    n_jobs=-1
)

final_real_hotel_model.fit(
    X_real_hotel_full,
    y_real_hotel_full,
    verbose=False
)

In [ ]:
hotel_full_data["is_placeholder"] = (
    hotel_full_data["price"] >= 30000
).astype(int)

X_hotel_class_full = hotel_full_data.drop(
    columns=["price", "is_placeholder"]
)

y_hotel_class_full = hotel_full_data["is_placeholder"]

final_placeholder_classifier = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    reg_alpha=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=12,
    tree_method="hist",
    n_jobs=-1
)

final_placeholder_classifier.fit(
    X_hotel_class_full,
    y_hotel_class_full,
    verbose=False
)

In [ ]:
placeholder_hotel_full_data = placeholder_hotel_full_data.copy()

placeholder_hotel_full_data["placeholder_price_class"] = (
    placeholder_hotel_full_data["price"] == 50000
).astype(int)

X_placeholder_full = placeholder_hotel_full_data.drop(
    columns=["price", "placeholder_price_class"]
)

y_placeholder_full = placeholder_hotel_full_data["placeholder_price_class"]

final_price_classifier = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=3,
    reg_alpha=0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=12,
    tree_method="hist",
    n_jobs=-1
)

final_price_classifier.fit(
    X_placeholder_full,
    y_placeholder_full,
    verbose=False
)

## 4) Exporting the Predictions

In [ ]:
test_predictions = np.zeros(len(X_test_encoded))

nonhotel_test_rows = X_test_encoded["is_hotel"] == 0
hotel_test_rows = X_test_encoded["is_hotel"] == 1

test_predictions[nonhotel_test_rows] = final_nonhotel_model.predict(
    X_test_encoded[nonhotel_test_rows]
)

X_test_hotels = X_test_encoded[hotel_test_rows].copy()

placeholder_probs = final_placeholder_classifier.predict_proba(
    X_test_hotels
)[:, 1]

hotel_placeholder_pred = (placeholder_probs >= 0.5).astype(int)

hotel_predictions = np.zeros(len(X_test_hotels))

real_hotel_rows = hotel_placeholder_pred == 0
placeholder_hotel_rows = hotel_placeholder_pred == 1

if real_hotel_rows.sum() > 0:
    hotel_predictions[real_hotel_rows] = final_real_hotel_model.predict(
        X_test_hotels[real_hotel_rows]
    )

if placeholder_hotel_rows.sum() > 0:
    placeholder_price_class = final_price_classifier.predict(
        X_test_hotels[placeholder_hotel_rows]
    )

    placeholder_price_pred = np.where(
        placeholder_price_class == 1,
        50000,
        40000
    )

    hotel_predictions[placeholder_hotel_rows] = placeholder_price_pred

test_predictions[hotel_test_rows] = hotel_predictions

test_predictions = np.maximum(test_predictions, 0)

submission = pd.DataFrame({
    "id": test_ids,
    "predicted": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()